In [ ]:
!pip install groq langgraph langchain-core python-whois requests beautifulsoup4 dnspython

In [25]:
import os
import json
import time
import requests
import whois
import dns.resolver
import re
from datetime import datetime, timezone
from typing import TypedDict
from bs4 import BeautifulSoup
from groq import Groq
from google.colab import userdata
from langgraph.graph import StateGraph, END

os.environ["GROQ_API_KEY"]       = userdata.get("GROQ_API_KEY")
os.environ["VIRUSTOTAL_API_KEY"] = userdata.get("VIRUSTOTAL_API_KEY")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
VT_KEY = os.environ["VIRUSTOTAL_API_KEY"]
MODEL  = "llama-3.3-70b-versatile"

print("✅ All imports loaded")
print("✅ Credentials ready")


✅ All imports loaded
✅ Credentials ready


In [26]:
# CONCEPT: One shared whiteboard all agents read and write.
# TypedDict enforces field names — prevents typos across agents.

class AgentState(TypedDict):
    original_message  : str
    orchestrator_plan : str
    entities_found    : dict
    nlp_findings      : dict
    technical_findings: dict
    cultural_findings : dict
    risk_score        : dict
    final_verdict     : dict
    agents_completed  : list
    error_log         : list



In [43]:
# CONCEPT: These are plain Python functions — not LLM calls.
# The Technical Specialist agent calls these directly in code.
# Phase 3 does NOT use a TOOLS list like Phase 2 did because
# the technical agent always runs ALL tools systematically
# rather than letting the LLM decide which ones to call.

# ── Tool 1: WHOIS Domain Lookup ──────────────────────────────
def whois_lookup(domain: str) -> dict:
    """
    Looks up domain registration data.
    Returns age, registrar, country, and risk level.
    Newly registered domains = strong scam signal.
    """
    print(f"    🔍 WHOIS: {domain}")
    try:
        w        = whois.whois(domain)
        creation = w.creation_date
        if isinstance(creation, list):
            creation = creation[0]
        if creation:
            if creation.tzinfo is None:
                creation = creation.replace(tzinfo=timezone.utc)
            age_days = (datetime.now(timezone.utc) - creation).days
        else:
            age_days = None

        if age_days is None:       age_risk = "unknown"
        elif age_days < 30:        age_risk = "VERY HIGH"
        elif age_days < 180:       age_risk = "HIGH"
        elif age_days < 365:       age_risk = "MEDIUM"
        else:                      age_risk = "LOW"

        result = {
            "domain"    : domain,
            "age_days"  : age_days,
            "age_risk"  : age_risk,
            "registrar" : str(w.registrar) if w.registrar else "unknown",
            "country"   : str(w.country)   if w.country   else "unknown",
            "status"    : "success"
        }
        print(f"      Age      : {age_days} days → risk: {age_risk}")
        print(f"      Country  : {result['country']}")
        return result

    except Exception as e:
        print(f"      ⚠ Failed: {e}")
        return {"domain": domain, "status": "failed", "error": str(e),
                "age_risk": "unknown"}

# ── Tool 2: VirusTotal URL Scanner ───────────────────────────
def virustotal_scan(url: str) -> dict:
    """
    Scans a URL against 70+ security engines via VirusTotal.
    Free tier: 4 requests/minute — we wait between calls.
    """
    print(f"    🔍 VirusTotal: {url[:60]}")
    headers = {"x-apikey": VT_KEY}
    try:
        r = requests.post(
            "https://www.virustotal.com/api/v3/urls",
            headers=headers, data={"url": url}, timeout=15
        )
        if r.status_code != 200:
            return {"url": url, "status": "failed",
                    "error": f"HTTP {r.status_code}"}

        analysis_id = r.json()["data"]["id"]
        print(f"      Submitted. Waiting for analysis...")
        time.sleep(5)

        r2    = requests.get(
            f"https://www.virustotal.com/api/v3/analyses/{analysis_id}",
            headers=headers, timeout=15
        )
        stats      = r2.json()["data"]["attributes"]["stats"]
        malicious  = stats.get("malicious",  0)
        suspicious = stats.get("suspicious", 0)
        total      = sum(stats.values())

        if malicious >= 5:   vt_risk = "VERY HIGH"
        elif malicious >= 2: vt_risk = "HIGH"
        elif malicious >= 1: vt_risk = "MEDIUM"
        else:                vt_risk = "LOW"

        print(f"      Malicious : {malicious}/{total} → risk: {vt_risk}")
        return {
            "url": url, "malicious": malicious,
            "suspicious": suspicious, "total_engines": total,
            "vt_risk": vt_risk, "status": "success"
        }
    except Exception as e:
        print(f"      ⚠ Failed: {e}")
        return {"url": url, "status": "failed", "error": str(e)}


# ── Tool 3: Website Scraper ───────────────────────────────────
def scrape_website(url: str) -> dict:
    """
    Visits a suspicious URL and checks content for scam patterns.
    Uses your BeautifulSoup skills directly.
    """
    print(f"    🔍 Scraping: {url[:60]}")
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r    = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text       = " ".join(soup.get_text().split())[:2000]
        text_lower = text.lower()

        flags = {
            "login_form"       : bool(soup.find("form")),
            "password_field"   : bool(soup.find("input", {"type": "password"})),
            "urgency_language" : any(w in text_lower for w in [
                "urgent", "expires", "act now",
                "within 24 hours", "immediately"]),
            "prize_language"   : any(w in text_lower for w in [
                "congratulations", "winner", "prize", "you have won"]),
            "financial_request": any(w in text_lower for w in [
                "bank account", "wire transfer",
                "processing fee", "western union"]),
        }
        found = [k for k, v in flags.items() if v]
        print(f"      Flags found : {found}")
        return {
            "url": url,
            "title": soup.title.string if soup.title else "none",
            "flags_found": found,
            "flags_count": len(found),
            "status": "success"
        }
    except requests.exceptions.ConnectionError:
        print(f"      ⚠ Unreachable — strong scam signal")
        return {"url": url, "status": "unreachable",
                "note": "Domain unreachable — strong scam signal"}
    except Exception as e:
        print(f"      ⚠ Failed: {e}")
        return {"url": url, "status": "failed", "error": str(e)}


# ── Tool 4: Email Domain Analyzer ────────────────────────────
def analyze_email_domain(email: str) -> dict:
    """
    Analyzes an email address for suspicious patterns.
    Checks free provider, suspicious prefix, MX records.
    """
    print(f"    🔍 Email analysis: {email}")
    if "@" not in email:
        return {"email": email, "status": "invalid"}

    prefix, domain = email.split("@", 1)
    free_providers = [
        "gmail.com", "yahoo.com", "hotmail.com", "outlook.com",
        "live.com", "protonmail.com", "icloud.com", "aol.com"
    ]
    suspicious_prefixes = [
        "hralert", "hr-alert", "noreply-hr", "jobs-alert",
        "alert-team", "recruitment-alert", "info-team", "admin-alert",
        "marketing", "no-reply", "donotreply", "noreply"
    ]

    is_free    = domain.lower() in free_providers
    sus_prefix = any(p in prefix.lower() for p in suspicious_prefixes)

    try:
        dns.resolver.resolve(domain, "MX")
        has_mx = True
    except Exception:
        has_mx = False

    signals = []
    if is_free:    signals.append("free email provider used for business")
    if sus_prefix: signals.append(f"suspicious prefix pattern: {prefix}")
    if not has_mx: signals.append("domain has no MX records")

    risk = "HIGH" if len(signals) >= 2 else "MEDIUM" if signals else "LOW"

    print(f"      Free provider    : {is_free}")
    print(f"      Suspicious prefix: {sus_prefix}")
    print(f"      Has MX records   : {has_mx}")
    print(f"      Email risk       : {risk}")

    return {
        "email": email, "domain": domain, "prefix": prefix,
        "is_free_provider": is_free, "suspicious_prefix": sus_prefix,
        "has_mx_records": has_mx, "risk_signals": signals,
        "email_risk": risk, "status": "success"
    }


# ── Tool 5: Domain Mismatch Checker ──────────────────────────
def check_domain_mismatch(sender_email: str,
                          body_domains: list) -> dict:
    """
    Checks if sender domain matches domains in the email body.
    Mismatch = domain mismatch attack — a deliberate scam technique.
    Added in Phase 2 after observing real two-domain scam emails.
    """
    print(f"    🔍 Domain mismatch check")
    if "@" not in sender_email:
        return {"status": "invalid"}

    sender_domain = sender_email.split("@")[1].lower()
    body_domains  = [d.lower().strip() for d in body_domains]
    exact_match   = sender_domain in body_domains
    mismatches    = [d for d in body_domains if d != sender_domain]

    if not exact_match and mismatches:
        risk = "HIGH — sender domain does not match body domains"
    else:
        risk = "LOW — domains are consistent"

    print(f"      Sender domain : {sender_domain}")
    print(f"      Body domains  : {body_domains}")
    print(f"      Mismatch risk : {risk}")

    return {
        "sender_domain": sender_domain,
        "body_domains" : body_domains,
        "exact_match"  : exact_match,
        "mismatches"   : mismatches,
        "mismatch_risk": risk,
        "status"       : "success"
    }


# ── Tool 6: Company Existence Checker ────────────
# CONCEPT: Sophisticated scams use professional emails from
# companies that barely exist. This tool checks whether the
# recruiting company has real verifiable online presence.
# Catches scams that pass all other checks because they look
# professionally written but the company is essentially fake.

def check_company_existence(company_name: str,
                            domain: str) -> dict:
    """
    Verifies whether a recruiting company actually exists online.
    Checks: name-domain match, website content, careers section,
    and whether the company name appears on its own website.

    CATCHES: Polished scam emails from fake or shell companies
    that avoid urgency language and unrealistic salary claims.
    """
    print(f"    🔍 Company existence: {company_name} / {domain}")

    signals = {
        "domain_checked"    : domain,
        "company_name"      : company_name,
        "name_domain_match" : False,
        "existence_signals" : [],
        "absence_signals"   : [],
        "existence_risk"    : "LOW"
    }

    # Check if company name words appear in the domain
    name_clean   = company_name.lower().replace(" ","").replace("-","")
    domain_root  = domain.lower().split(".")[0].replace("-","")

    if domain_root in name_clean or name_clean in domain_root:
        signals["name_domain_match"] = True
        signals["existence_signals"].append("company name matches domain")
    else:
        signals["absence_signals"].append(
            f"company '{company_name}' does not match domain '{domain}'"
        )

    # Try to fetch and inspect the company website
    try:
        headers  = {"User-Agent": "Mozilla/5.0"}
        r        = requests.get(
            f"https://{domain}", headers=headers, timeout=8
        )
        soup     = BeautifulSoup(r.text, "html.parser")
        text     = soup.get_text().lower()

        # Check for basic business content
        if any(w in text for w in ["about us", "our team",
                                    "contact", "services"]):
            signals["existence_signals"].append(
                "website has basic business content")
        else:
            signals["absence_signals"].append(
                "website lacks basic business content")

        # Check for careers/jobs section
        if any(w in text for w in ["careers", "jobs",
                                    "vacancies", "hiring"]):
            signals["existence_signals"].append(
                "website has careers section")
        else:
            signals["absence_signals"].append(
                "no careers section found on website")

        # Check if company name appears on its own site
        name_words = [w for w in company_name.lower().split()
                      if len(w) > 4]
        if any(w in text for w in name_words):
            signals["existence_signals"].append(
                "company name found on website")
        else:
            signals["absence_signals"].append(
                "company name not found on its own website")

    except requests.exceptions.ConnectionError:
        signals["absence_signals"].append(
            "company website completely unreachable")
    except Exception as e:
        signals["absence_signals"].append(
            f"website check failed: {str(e)[:60]}")

    # Calculate existence risk from absence signals
    # Website unreachable is a strong standalone signal
    # Override to HIGH regardless of name match
    website_unreachable = any(
        "unreachable" in s for s in signals["absence_signals"]
    )

    absence_count = len(signals["absence_signals"])

    if website_unreachable and absence_count >= 2:
        signals["existence_risk"] = "HIGH"
    elif website_unreachable or absence_count >= 3:
        signals["existence_risk"] = "HIGH"
    elif absence_count >= 2:
        signals["existence_risk"] = "MEDIUM"
    else:
        signals["existence_risk"] = "LOW"

    print(f"      Existence signals : {signals['existence_signals']}")
    print(f"      Absence signals   : {signals['absence_signals']}")
    print(f"      Existence risk    : {signals['existence_risk']}")

    return signals

#------------------------Extract Entities -----------------------#

def extract_entities_python(message: str) -> dict:
    """
    Extracts emails, URLs, and domains using Python regex.
    More reliable than asking the LLM to extract them.
    LLMs sometimes miss entities — Python regex never does.
    """
    # Extract email addresses
    emails = re.findall(
        r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}',
        message
    )

    # Extract URLs
    urls = re.findall(
        r'https?://[^\s<>"{}|\\^`\[\]]+',
        message
    )

    # Extract domains from emails
    email_domains = list(set([
        e.split("@")[1] for e in emails if "@" in e
    ]))

    # Extract domains from URLs
    url_domains = []
    for url in urls:
        try:
            from urllib.parse import urlparse
            domain = urlparse(url).netloc
            if domain:
                url_domains.append(domain)
        except:
            pass

    all_domains = list(set(email_domains + url_domains))

    print(f"    Python extractor found:")
    print(f"    Emails  : {emails}")
    print(f"    URLs    : {urls}")
    print(f"    Domains : {all_domains}")

    return {
        "emails" : emails,
        "urls"   : urls,
        "domains": all_domains,
    }

In [28]:
def call_llm(system_prompt: str, user_message: str,
             temperature: float = 0.1) -> str:
    """Calls the LLM and returns text response."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=temperature,
        max_tokens=1500,
    )
    return response.choices[0].message.content.strip()


def parse_json_response(raw: str) -> dict:
    """Safely parses JSON from LLM response."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        return {"error": "Could not parse JSON", "raw": raw}



In [29]:
def orchestrator_agent(state: AgentState) -> AgentState:
    print("\n" + "="*60)
    print("  AGENT 1: ORCHESTRATOR")
    print("="*60)

    # STEP 1: Extract entities with Python — reliable, never misses
    python_entities = extract_entities_python(state["original_message"])

    # STEP 2: Ask LLM only for interpretation — not extraction
    system_prompt = """
    You are the lead fraud investigator.
    Entities have already been extracted by Python tools.
    Your job is to interpret the message context only.

    ## IDENTITY CONSISTENCY CHECK:
    1. Are multiple company names present in one sender identity?
       e.g. "ZaviyarGroup | WAS Group | FastInsu" → SUSPICIOUS
    2. Does the requested action match legitimate recruitment?
       "Book a slot", "first come first served" → mass spam not real hiring
    3. Does the email domain match the claimed company name?

    Return ONLY this JSON:
    {
      "claimed_identity" : "who claims to be sending this",
      "requested_action" : "what they want the recipient to do",
      "identity_flags"   : ["list of identity inconsistencies"],
      "identity_risk"    : "HIGH or MEDIUM or LOW",
      "investigation_plan": "one sentence on what to investigate",
      "priority"         : "HIGH or MEDIUM or LOW"
    }
    """

    raw    = call_llm(system_prompt,
                      f"Interpret this message:\n\n{state['original_message']}")
    result = parse_json_response(raw)

    # STEP 3: Merge Python entities + LLM interpretation
    final_entities = {
        "emails"           : python_entities["emails"],
        "urls"             : python_entities["urls"],
        "domains"          : python_entities["domains"],
        "claimed_identity" : result.get("claimed_identity", ""),
        "requested_action" : result.get("requested_action", ""),
        "identity_flags"   : result.get("identity_flags", []),
        "identity_risk"    : result.get("identity_risk", "LOW"),
    }

    print(f"  Claimed identity : {final_entities['claimed_identity']}")
    print(f"  Identity flags   : {final_entities['identity_flags']}")
    print(f"  Identity risk    : {final_entities['identity_risk']}")
    print(f"  Priority         : {result.get('priority', 'unknown')}")

    state["entities_found"]    = final_entities
    state["orchestrator_plan"] = result.get("investigation_plan", "")
    state["agents_completed"]  = ["orchestrator"]
    return state

In [30]:
def nlp_specialist_agent(state: AgentState) -> AgentState:
    """
    ROLE: Language expert. Analyzes text for psychological
    manipulation, urgency tactics, threats, and deceptive patterns.
    Text only — never touches URLs or domains.
    """
    print("\n" + "="*60)
    print("  AGENT 2: NLP SPECIALIST")
    print("="*60)

    system_prompt = """
    You are an expert in linguistic analysis and psychological
    manipulation detection. Analyze ONLY the text content —
    not URLs or domains.

    Look for:
    1. Urgency language ("act now", "expires in 24 hours")
    2. Threat language ("account will be closed", "legal action")
    3. Authority impersonation ("official government", "bank security")
    4. Reward manipulation ("you have won", "selected for")
    5. Information harvesting ("send your ID", "confirm bank details")
    6. Unusual grammar suggesting non-native templates
    7. Generic greetings ("Dear Candidate" instead of a real name)
    8. Mass recruitment language ("book a slot", "first come first served",
       "limited slots available") — signals bulk spam not real hiring

    Return ONLY this JSON:
    {
      "urgency_found"      : true or false,
      "threat_found"       : true or false,
      "authority_claim"    : true or false,
      "reward_claim"       : true or false,
      "info_harvesting"    : true or false,
      "generic_greeting"   : true or false,
      "grammar_issues"     : true or false,
      "mass_recruitment"   : true or false,
      "key_phrases"        : ["suspicious phrases found"],
      "sentiment"          : "aggressive / friendly / neutral / urgent",
      "manipulation_score" : 0 to 100,
      "nlp_risk"           : "HIGH or MEDIUM or LOW",
      "nlp_summary"        : "one sentence summary"
    }
    """

    raw    = call_llm(system_prompt,
                      f"Analyze:\n\n{state['original_message']}")
    result = parse_json_response(raw)

    # RULE-BASED RISK OVERRIDE
    # Do not trust LLM to self-assess risk — calculate it from its own findings
    high_nlp_signals = sum([
        bool(result.get("urgency_found")),
        bool(result.get("threat_found")),
        bool(result.get("authority_claim")),
        bool(result.get("reward_claim")),
        bool(result.get("info_harvesting")),
        bool(result.get("mass_recruitment")),
        bool(result.get("generic_greeting")),
    ])

    if high_nlp_signals >= 4:        result["nlp_risk"] = "HIGH"
    elif high_nlp_signals >= 2:      result["nlp_risk"] = "MEDIUM"
    else:                            result["nlp_risk"] = "LOW"

    result["high_nlp_signals"] = high_nlp_signals
    print(f"  High NLP signals : {high_nlp_signals} → risk: {result['nlp_risk']}")

    state["nlp_findings"] = result
    state["agents_completed"].append("nlp_specialist")
    return state


In [31]:
# 1. Now calls check_company_existence() on claimed identity
# 2. Compensating control added — if no URLs but cultural+NLP HIGH,
#    penalize score (sophisticated scammers avoid URLs deliberately)
# 3. Identity risk from orchestrator now feeds into scoring

def technical_specialist_agent(state: AgentState) -> AgentState:
    """
    ROLE: Technical investigator. Runs all verification tools
    against extracted entities and produces evidence report.
    """
    print("\n" + "="*60)
    print("  AGENT 3: TECHNICAL SPECIALIST")
    print("="*60)

    entities = state.get("entities_found", {})
    evidence = {
        "whois_results"    : [],
        "virustotal"       : [],
        "email_analysis"   : [],
        "website_scrapes"  : [],
        "company_existence": {},
        "domain_mismatch"  : {},
        "technical_risk"   : "LOW"
    }

    # Extract all domains including from email addresses
    domains       = entities.get("domains", [])
    emails        = entities.get("emails",  [])
    urls          = entities.get("urls",    [])
    email_domains = [e.split("@")[1] for e in emails if "@" in e]
    all_domains   = list(set(domains + email_domains))

    # ── Run WHOIS on all domains ──────────────────────────────
    for domain in all_domains:
        result = whois_lookup(domain)
        evidence["whois_results"].append(result)
        time.sleep(1)

    # ── Run email domain analysis ─────────────────────────────
    for email in emails:
        result = analyze_email_domain(email)
        evidence["email_analysis"].append(result)

    # ── Check domain mismatch if multiple domains found ───────
    if emails and len(all_domains) > 1:
        result = check_domain_mismatch(
            sender_email=emails[0],
            body_domains=[d for d in all_domains
                          if d not in emails[0]]
        )
        evidence["domain_mismatch"] = result

    # ── Run company existence check (NEW in v2) ───────────────
    claimed_identity = entities.get("claimed_identity", "")
    if claimed_identity and all_domains:
        result = check_company_existence(
            company_name=claimed_identity,
            domain=all_domains[0]
        )
        evidence["company_existence"] = result

    # ── Run VirusTotal on URLs (max 2 for free tier) ──────────
    for url in urls[:2]:
        result = virustotal_scan(url)
        evidence["virustotal"].append(result)
        time.sleep(15)

    # ── Scrape websites (max 1 to save time) ─────────────────
    for url in urls[:1]:
        result = scrape_website(url)
        evidence["website_scrapes"].append(result)

    # ── Calculate overall technical risk ─────────────────────
    # CONCEPT: Aggregate risk — count high-risk signals across all tools
    high_signals = 0

    for w in evidence["whois_results"]:
        if w.get("age_risk") in ["VERY HIGH", "HIGH"]:
            high_signals += 1

    for e in evidence["email_analysis"]:
        if e.get("email_risk") == "HIGH":
            high_signals += 1

    for v in evidence["virustotal"]:
        if v.get("vt_risk") in ["VERY HIGH", "HIGH"]:
            high_signals += 2

    for s in evidence["website_scrapes"]:
        if s.get("flags_count", 0) >= 2:
            high_signals += 1

    # Company existence penalty (NEW in v2)
    company_risk = evidence["company_existence"].get(
        "existence_risk", "LOW")
    if company_risk == "HIGH":
        high_signals += 2
        print("  ⚠ Company existence HIGH risk — adding 2 signals")
    elif company_risk == "MEDIUM":
        high_signals += 1
        print("  ⚠ Company existence MEDIUM risk — adding 1 signal")

    # Domain mismatch penalty
    mismatch_risk = evidence["domain_mismatch"].get(
        "mismatch_risk", "")
    if "HIGH" in str(mismatch_risk):
        high_signals += 1
        print("  ⚠ Domain mismatch detected — adding 1 signal")

    # Identity risk from orchestrator
    identity_risk = entities.get("identity_risk", "LOW")
    if identity_risk == "HIGH":
        high_signals += 1
        print("  ⚠ Identity inconsistency HIGH — adding 1 signal")

    # COMPENSATING CONTROL (NEW in v2):
    # CONCEPT: Sophisticated scammers deliberately avoid including
    # URLs so VirusTotal cannot flag them. No URLs + high NLP/cultural
    # signals is itself suspicious — we penalize this pattern.
    nlp_risk      = state.get("nlp_findings", {}).get("nlp_risk", "LOW")
    cultural_risk = state.get("cultural_findings", {}).get(
        "cultural_risk", "LOW")

    if (not urls and
        nlp_risk      in ["HIGH", "VERY HIGH"] and
        cultural_risk in ["HIGH", "VERY HIGH"]):
        high_signals += 2
        evidence["technical_risk"] = "HIGH"
        print("\n  ⚠ COMPENSATING CONTROL TRIGGERED:")
        print("    No URLs found but NLP + Cultural both HIGH")
        print("    → Sophisticated scam avoiding technical detection")
        print("    → Adding 2 penalty signals")

    # Final technical risk level
    if high_signals >= 4:   evidence["technical_risk"] = "VERY HIGH"
    elif high_signals >= 3: evidence["technical_risk"] = "HIGH"
    elif high_signals >= 2: evidence["technical_risk"] = "MEDIUM"
    elif high_signals >= 1: evidence["technical_risk"] = "LOW-MEDIUM"
    else:                   evidence["technical_risk"] = "LOW"

    print(f"\n  Total high signals : {high_signals}")
    print(f"  Technical risk     : {evidence['technical_risk']}")

    state["technical_findings"] = evidence
    state["agents_completed"].append("technical_specialist")
    return state


In [32]:
# AMENDMENT v2: Extended with unsolicited recruitment patterns.
# Now catches sophisticated job scams that avoid obvious red flags
# but use mass recruitment language and unverifiable companies.

def cultural_context_agent(state: AgentState) -> AgentState:
    """
    ROLE: UAE cultural expert. Detects UAE-specific scam patterns
    that generic systems miss — local salary norms, known targets,
    Arabic language patterns, unsolicited recruitment tactics.
    """
    print("\n" + "="*60)
    print("  AGENT 4: CULTURAL CONTEXT SPECIALIST")
    print("="*60)

    system_prompt = """
    You are a UAE-based fraud specialist with deep knowledge of scams
    targeting people in the UAE, GCC, and South Asian expat communities.

    ## KNOWN UAE SCAM PATTERNS:

    JOB SCAMS:
    - Fake hiring from ADNOC, Emirates Group, Etisalat, du, DEWA, RTA
    - Unrealistic salaries (AED 15,000+ for entry level roles)
    - Requests for Emirates ID before any face-to-face interview
    - Asking for "visa processing fees" or "medical test fees" upfront
    - Companies claiming presence in Dubai Internet City or DIFC
      without verifiable registration

    UNSOLICITED RECRUITMENT RED FLAGS (NEW):
    - Unsolicited contact from companies the recipient never applied to
    - Multiple company names representing one sender
      e.g. "ZaviyarGroup | WAS Group | FastInsu" — real companies
      use ONE name
    - Mass recruitment language: "book a slot", "first come first served",
      "limited slots" — real hiring is targeted, not bulk
    - No mention of specific job requirements or qualifications
    - No reference to where they found your CV or profile
    - Insurance or financial companies recruiting for unrelated roles
    - Generic role titles with no technical or experience requirements
    If 3 or more unsolicited recruitment flags present → cultural_risk: HIGH

    FINANCIAL SCAMS:
    - Fake Dubai Government lucky draws
    - Bogus UAE Central Bank notifications
    - Fake ENOC / DEWA / Salik refunds

    IMPERSONATION:
    - UAE Embassy or consulate communications
    - Dubai Police or Abu Dhabi Police
    - Major UAE banks: Emirates NBD, FAB, ADCB, Mashreq

    ARABIC SCAM PHRASES:
    - عاجل (urgent), فوري (immediate)
    - لقد فزت (you have won), جائزة كبرى (grand prize)
    - سيتم إغلاق حسابك (your account will be closed)

    Return ONLY this JSON:
    {
      "uae_entity_impersonated"    : "name or null",
      "known_uae_scam_pattern"     : true or false,
      "scam_pattern_name"          : "pattern name or null",
      "unrealistic_offer"          : true or false,
      "offer_details"              : "details or null",
      "unsolicited_recruitment"    : true or false,
      "unsolicited_flags_found"    : ["list of unsolicited flags"],
      "multiple_company_names"     : true or false,
      "mass_recruitment_language"  : true or false,
      "arabic_scam_phrases"        : ["Arabic phrases found if any"],
      "targets_expat_community"    : true or false,
      "cultural_red_flags"         : ["all UAE-specific red flags"],
      "cultural_risk"              : "HIGH or MEDIUM or LOW",
      "cultural_summary"           : "one sentence summary"
    }
    """

    raw    = call_llm(system_prompt,
                      f"Analyze for UAE context:\n\n{state['original_message']}")
    result = parse_json_response(raw)

    print(f"  UAE entity impersonated    : {result.get('uae_entity_impersonated')}")
    print(f"  Unsolicited recruitment    : {result.get('unsolicited_recruitment')}")
    print(f"  Multiple company names     : {result.get('multiple_company_names')}")
    print(f"  Mass recruitment language  : {result.get('mass_recruitment_language')}")
    print(f"  Cultural risk              : {result.get('cultural_risk')}")
    print(f"  Cultural flags             : {result.get('cultural_red_flags', [])}")

    state["cultural_findings"] = result
    state["agents_completed"].append("cultural_context")
    return state


In [33]:
# CONCEPT: Weighted scoring system.
# Technical = 50% (hard facts), Cultural = 30% (context),
# NLP = 20% (linguistic signals)

def risk_scorer_agent(state: AgentState) -> AgentState:
    """
    ROLE: Quantitative analyst. Weighs all specialist findings
    and produces a calibrated 0-100 risk score with breakdown.
    """
    print("\n" + "="*60)
    print("  AGENT 5: RISK SCORER")
    print("="*60)

    # ADAPTIVE WEIGHTS based on available evidence
    # If no technical evidence exists, redistribute its weight
    tech_risk = state.get("technical_findings", {}).get(
        "technical_risk", "LOW")
    has_technical_evidence = bool(
        state.get("technical_findings", {}).get("whois_results") or
        state.get("technical_findings", {}).get("email_analysis") or
        state.get("technical_findings", {}).get("company_existence")
    )

    if has_technical_evidence:
        # Normal weights — we have hard evidence
        weights = {"technical": 0.50, "cultural": 0.30, "nlp": 0.20}
        print("  Using standard weights (technical evidence available)")
    else:
        # No technical evidence — redistribute to cultural and NLP
        # This handles sophisticated text-only scams
        weights = {"technical": 0.20, "cultural": 0.50, "nlp": 0.30}
        print("  ⚠ No technical evidence — using adaptive weights")
        print("    Cultural: 50%, NLP: 30%, Technical: 20%")

    def risk_to_score(risk: str) -> int:
        return {
            "VERY HIGH" : 95,
            "HIGH"      : 75,
            "LOW-MEDIUM": 55,
            "MEDIUM"    : 50,
            "LOW"       : 15,
            "unknown"   : 30,
        }.get(risk, 30)

    tech_risk     = state.get("technical_findings", {}).get(
        "technical_risk", "LOW")
    cultural_risk = state.get("cultural_findings",  {}).get(
        "cultural_risk",  "LOW")
    nlp_risk      = state.get("nlp_findings",       {}).get(
        "nlp_risk",       "LOW")

    tech_score     = risk_to_score(tech_risk)
    cultural_score = risk_to_score(cultural_risk)
    nlp_score      = risk_to_score(nlp_risk)

    final_score = int(
        tech_score     * weights["technical"] +
        cultural_score * weights["cultural"]  +
        nlp_score      * weights["nlp"]
    )

    if final_score >= 85:   confidence = "VERY HIGH"
    elif final_score >= 65: confidence = "HIGH"
    elif final_score >= 45: confidence = "MEDIUM"
    else:                   confidence = "LOW"

    risk_score = {
        "final_score"     : final_score,
        "confidence"      : confidence,
        "is_scam"         : final_score >= 55,
        "breakdown"       : {
            "technical_score" : tech_score,
            "cultural_score"  : cultural_score,
            "nlp_score"       : nlp_score,
            "technical_weight": "50%",
            "cultural_weight" : "30%",
            "nlp_weight"      : "20%",
        },
        "individual_risks": {
            "technical_risk": tech_risk,
            "cultural_risk" : cultural_risk,
            "nlp_risk"      : nlp_risk,
        }
    }

    print(f"  Technical  : {tech_score} × 50% = {int(tech_score*0.5)}")
    print(f"  Cultural   : {cultural_score} × 30% = {int(cultural_score*0.3)}")
    print(f"  NLP        : {nlp_score} × 20% = {int(nlp_score*0.2)}")
    print(f"  {'─'*35}")
    print(f"  Final score: {final_score}/100")
    print(f"  Confidence : {confidence}")
    print(f"  Is scam    : {risk_score['is_scam']}")

    state["risk_score"] = risk_score
    state["agents_completed"].append("risk_scorer")
    return state


In [34]:
def verdict_agent(state: AgentState) -> AgentState:
    """
    ROLE: Orchestrator's final step. Synthesizes all agent
    findings into one clean human-readable verdict report.
    """
    print("\n" + "="*60)
    print("  AGENT 6: FINAL VERDICT")
    print("="*60)

    all_findings = {
        "original_message"  : state["original_message"][:300],
        "entities_found"    : state.get("entities_found",     {}),
        "nlp_findings"      : state.get("nlp_findings",       {}),
        "technical_findings": state.get("technical_findings", {}),
        "cultural_findings" : state.get("cultural_findings",  {}),
        "risk_score"        : state.get("risk_score",         {}),
    }

    system_prompt = """
    You are the lead fraud investigator writing the final case report.
    Synthesize findings from four specialist agents into a clear verdict.

    Return ONLY this JSON:
    {
      "is_scam"           : true or false,
      "confidence"        : 0 to 100,
      "scam_type"         : one of ["phishing","lottery_fraud","advance_fee",
                            "romance_scam","investment_fraud","impersonation",
                            "job_scam","tech_support_scam","not_a_scam","unknown"],
      "verdict_summary"   : "2-3 sentence plain English verdict",
      "key_evidence"      : ["top 3-5 pieces of evidence that decided verdict"],
      "red_flags"         : ["all red flags across all agents"],
      "safe_indicators"   : ["any legitimate signals found"],
      "recommended_action": "clear action for the recipient",
      "agents_report"     : {
        "nlp"      : "one line NLP summary",
        "technical": "one line technical summary",
        "cultural" : "one line cultural summary",
        "score"    : "risk score and breakdown summary"
      }
    }
    """

    raw    = call_llm(system_prompt, json.dumps(all_findings, default=str))
    result = parse_json_response(raw)

    # Override with our calculated score — do not let LLM guess
    if state.get("risk_score"):
        result["confidence"] = state["risk_score"]["final_score"]
        result["is_scam"]    = state["risk_score"]["is_scam"]

    state["final_verdict"] = result
    state["agents_completed"].append("verdict")
    return state



In [35]:
# CONCEPT: Wire all agents as nodes connected by edges.
# Graph enforces execution order and state passing between agents.

def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("orchestrator", orchestrator_agent)
    graph.add_node("nlp",          nlp_specialist_agent)
    graph.add_node("technical",    technical_specialist_agent)
    graph.add_node("cultural",     cultural_context_agent)
    graph.add_node("risk_scorer",  risk_scorer_agent)
    graph.add_node("verdict",      verdict_agent)

    graph.set_entry_point("orchestrator")
    graph.add_edge("orchestrator", "nlp")
    graph.add_edge("nlp",          "technical")
    graph.add_edge("technical",    "cultural")
    graph.add_edge("cultural",     "risk_scorer")
    graph.add_edge("risk_scorer",  "verdict")
    graph.add_edge("verdict",      END)

    return graph.compile()


investigation_graph = build_graph()
print("✅ Multi-agent graph compiled — v2 with all amendments")
print("   Flow: orchestrator → nlp → technical → cultural → scorer → verdict")
print("   New tools: company existence checker, identity consistency")
print("   New controls: compensating control, unsolicited recruitment")



✅ Multi-agent graph compiled — v2 with all amendments
   Flow: orchestrator → nlp → technical → cultural → scorer → verdict
   New tools: company existence checker, identity consistency
   New controls: compensating control, unsolicited recruitment


In [36]:
def display_verdict(state: AgentState):
    verdict    = state.get("final_verdict", {})
    risk_score = state.get("risk_score",    {})
    is_scam    = verdict.get("is_scam", False)
    confidence = verdict.get("confidence", 0)
    filled     = int(confidence / 10)
    bar        = "█" * filled + "░" * (10 - filled)

    print(f"\n{'─'*60}")
    print(f"  VERDICT   : {'🚨 SCAM DETECTED' if is_scam else '✅ LIKELY SAFE'}")
    print(f"  Score     : [{bar}] {confidence}/100")
    print(f"  Scam type : {verdict.get('scam_type','').replace('_',' ').title()}")
    print(f"  Agents    : {len(state.get('agents_completed',[]))} completed")
    print(f"{'─'*60}")

    breakdown = risk_score.get("breakdown", {})
    if breakdown:
        print(f"\n  📊 SCORE BREAKDOWN:")
        print(f"     Technical (50%) : {breakdown.get('technical_score', 0)}")
        print(f"     Cultural  (30%) : {breakdown.get('cultural_score',  0)}")
        print(f"     NLP       (20%) : {breakdown.get('nlp_score',       0)}")

    agents_report = verdict.get("agents_report", {})
    if agents_report:
        print(f"\n  🤖 AGENT REPORTS:")
        for agent, summary in agents_report.items():
            print(f"     {agent.upper():12}: {summary}")

    key_evidence = verdict.get("key_evidence", [])
    if key_evidence:
        print(f"\n  🔬 KEY EVIDENCE:")
        for e in key_evidence:
            print(f"     • {e}")

    red_flags = verdict.get("red_flags", [])
    if red_flags:
        print(f"\n  🚩 RED FLAGS:")
        for f in red_flags:
            print(f"     • {f}")

    print(f"\n  📋 SUMMARY:")
    print(f"     {verdict.get('verdict_summary', '')}")
    print(f"\n  💡 ACTION:")
    print(f"     {verdict.get('recommended_action', '')}")
    print(f"{'─'*60}\n")


In [37]:
def investigate(message: str) -> AgentState:
    initial_state: AgentState = {
        "original_message"  : message,
        "orchestrator_plan" : "",
        "entities_found"    : {},
        "nlp_findings"      : {},
        "technical_findings": {},
        "cultural_findings" : {},
        "risk_score"        : {},
        "final_verdict"     : {},
        "agents_completed"  : [],
        "error_log"         : [],
    }

    print("\n" + "="*60)
    print("  MULTI-AGENT ANTI-SCAM INVESTIGATION v2")
    print("  6 agents · 6 tools · UAE cultural context")
    print("="*60)

    return investigation_graph.invoke(initial_state)


In [38]:
test_messages = [

    # TEST 1: Sophisticated low-signal scam (previously scored 43%)
    # v2 should now catch this via company existence + identity checks
    """
    From: marketing@zaviyarhayatgroup.com
    Subject: Interview Invitation

    Dear Candidate,
    ZaviyarHayat Group | WAS Group | FastInsu is currently hiring.
    We would like to invite you for interview sessions.
    BOOK YOUR SLOT HERE — first come, first served.
    Recruitment Team
    """,

    # TEST 2: UAE job scam with ADNOC impersonation
    # Should trigger cultural + NLP agents strongly
    """
    From: hralert@wadialsagroup.com
    Subject: Urgent Interview — ADNOC Contractor

    Dear Candidate,
    You have been selected for an interview at ADNOC contractor
    Wadi Al Salam Group. Salary AED 18,000/month tax free.
    Interview Monday 10am. Bring Emirates ID and pay AED 300
    processing fee upon arrival.
    HR Department
    """,

    # TEST 3: Bank phishing with suspicious URL
    """
    Dear Emirates NBD Customer,
    Your account requires immediate verification.
    Click: http://emiratesnbd-secure-login.tk/verify
    Failure to verify within 2 hours will result in suspension.
    """,

    # TEST 4: Legitimate message — should remain safe
    """
    Hi, your Noon order #UAE-2847361 has been shipped.
    Expected delivery: tomorrow between 2-6pm.
    Track here: https://noon.com/uae/track/2847361
    Noon Customer Service
    """,
]

results = []
for i, message in enumerate(test_messages, 1):
    print(f"\n{'='*60}")
    print(f"  TEST {i} of {len(test_messages)}")
    print(f"{'='*60}")
    final_state = investigate(message)
    display_verdict(final_state)
    results.append(final_state)
    time.sleep(2)



  TEST 1 of 4

  MULTI-AGENT ANTI-SCAM INVESTIGATION v2
  6 agents · 6 tools · UAE cultural context

  AGENT 1: ORCHESTRATOR
    Python extractor found:
    Emails  : ['marketing@zaviyarhayatgroup.com']
    URLs    : []
    Domains : ['zaviyarhayatgroup.com']
  Claimed identity : ZaviyarHayat Group | WAS Group | FastInsu
  Identity flags   : ['Multiple company names in sender identity', 'Requested action does not match legitimate recruitment']
  Identity risk    : HIGH
  Priority         : HIGH

  AGENT 2: NLP SPECIALIST
  High NLP signals : 3 → risk: MEDIUM

  AGENT 3: TECHNICAL SPECIALIST
    🔍 WHOIS: zaviyarhayatgroup.com
      Age      : 64 days → risk: HIGH
      Country  : US
    🔍 Email analysis: marketing@zaviyarhayatgroup.com
      Free provider    : False
      Suspicious prefix: True
      Has MX records   : True
      Email risk       : MEDIUM
    🔍 Company existence: ZaviyarHayat Group | WAS Group | FastInsu / zaviyarhayatgroup.com
      Existence signals : ['company name

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


      Age      : None days → risk: unknown
      Country  : unknown
    🔍 Company existence: Emirates NBD / emiratesnbd-secure-login.tk
      Existence signals : ['company name matches domain']
      Absence signals   : ['company website completely unreachable']
      Existence risk    : LOW
    🔍 VirusTotal: http://emiratesnbd-secure-login.tk/verify
      Submitted. Waiting for analysis...
      Malicious : 0/0 → risk: LOW
    🔍 Scraping: http://emiratesnbd-secure-login.tk/verify
      ⚠ Unreachable — strong scam signal
  ⚠ Identity inconsistency HIGH — adding 1 signal

  Total high signals : 1
  Technical risk     : LOW-MEDIUM

  AGENT 4: CULTURAL CONTEXT SPECIALIST
  UAE entity impersonated    : Emirates NBD
  Unsolicited recruitment    : False
  Multiple company names     : False
  Mass recruitment language  : False
  Cultural risk              : HIGH
  Cultural flags             : ['IMPERSONATION of major UAE bank', 'URGENT action required', 'suspension threat']

  AGENT 5: RISK S

In [44]:
print("\n--- YOUR TURN ---")
your_message = """
ZaviyarhayatGroup <marketing@zaviyarhayatgroup.com>
Wed, Apr 22, 5:57 PM (9 days ago)
to me

HiKhan,

I’m sending over the updated list of locations and times for the upcoming interview sessions in Dubai and the wider GCC.
We have refreshed the schedule with direct contact details for the hiring teams. You can find the specific venue addresses and times for this week at the link below:

BOOK YOUR SLOT HERE

Quick notes for this week:
1.      Most sessions are first-come, first-served due to venue capacity.
2.      Direct employer contacts are included in the listings.
3.      There are no costs or fees to attend these sessions.

I hope this helps with your search.

Best,
Recruitment Team WAS Group | FastInsu
"""
your_state = investigate(your_message)
display_verdict(your_state)


--- YOUR TURN ---

  MULTI-AGENT ANTI-SCAM INVESTIGATION v2
  6 agents · 6 tools · UAE cultural context

  AGENT 1: ORCHESTRATOR
    Python extractor found:
    Emails  : ['marketing@zaviyarhayatgroup.com']
    URLs    : []
    Domains : ['zaviyarhayatgroup.com']
  Claimed identity : ZaviyarhayatGroup and Recruitment Team WAS Group | FastInsu
  Identity flags   : ['Multiple company names in sender identity', 'Email domain does not match claimed company name']
  Identity risk    : HIGH
  Priority         : HIGH

  AGENT 2: NLP SPECIALIST
  High NLP signals : 2 → risk: MEDIUM

  AGENT 3: TECHNICAL SPECIALIST
    🔍 WHOIS: zaviyarhayatgroup.com
      Age      : 64 days → risk: HIGH
      Country  : US
    🔍 Email analysis: marketing@zaviyarhayatgroup.com
      Free provider    : False
      Suspicious prefix: True
      Has MX records   : True
      Email risk       : MEDIUM
    🔍 Company existence: ZaviyarhayatGroup and Recruitment Team WAS Group | FastInsu / zaviyarhayatgroup.com
      

In [45]:
# ── DIAGNOSTIC CELL — run after investigate() ────────────────
# This shows exactly what each agent returned
# so we can see where the score is bleeding out

state = your_state   # or results[0] for Test 1

print("\n" + "="*60)
print("  DIAGNOSTIC REPORT")
print("="*60)

# Orchestrator findings
entities = state.get("entities_found", {})
print(f"\n  ORCHESTRATOR:")
print(f"  Identity flags : {entities.get('identity_flags', [])}")
print(f"  Identity risk  : {entities.get('identity_risk', 'not found')}")

# NLP findings
nlp = state.get("nlp_findings", {})
print(f"\n  NLP AGENT:")
print(f"  NLP risk         : {nlp.get('nlp_risk', 'not found')}")
print(f"  Manipulation     : {nlp.get('manipulation_score', 'not found')}")
print(f"  Mass recruitment : {nlp.get('mass_recruitment', 'not found')}")
print(f"  Urgency found    : {nlp.get('urgency_found', 'not found')}")

# Technical findings
tech = state.get("technical_findings", {})
print(f"\n  TECHNICAL AGENT:")
print(f"  Technical risk   : {tech.get('technical_risk', 'not found')}")
print(f"  WHOIS results    : {tech.get('whois_results', [])}")
print(f"  Email analysis   : {tech.get('email_analysis', [])}")
print(f"  Company existence: {tech.get('company_existence', {})}")
print(f"  Domain mismatch  : {tech.get('domain_mismatch', {})}")

# Cultural findings
cultural = state.get("cultural_findings", {})
print(f"\n  CULTURAL AGENT:")
print(f"  Cultural risk           : {cultural.get('cultural_risk', 'not found')}")
print(f"  Unsolicited recruitment : {cultural.get('unsolicited_recruitment', 'not found')}")
print(f"  Multiple company names  : {cultural.get('multiple_company_names', 'not found')}")
print(f"  Mass recruitment lang   : {cultural.get('mass_recruitment_language', 'not found')}")
print(f"  Unsolicited flags       : {cultural.get('unsolicited_flags_found', [])}")

# Risk score breakdown
risk = state.get("risk_score", {})
print(f"\n  RISK SCORER:")
print(f"  Final score      : {risk.get('final_score', 'not found')}")
print(f"  Breakdown        : {risk.get('breakdown', {})}")
print(f"  Individual risks : {risk.get('individual_risks', {})}")


  DIAGNOSTIC REPORT

  ORCHESTRATOR:
  Identity flags : ['Multiple company names in sender identity', 'Email domain does not match claimed company name']
  Identity risk  : HIGH

  NLP AGENT:
  NLP risk         : MEDIUM
  Manipulation     : 20
  Mass recruitment : True
  Urgency found    : True

  TECHNICAL AGENT:
  Technical risk   : VERY HIGH
  WHOIS results    : [{'domain': 'zaviyarhayatgroup.com', 'age_days': 64, 'age_risk': 'HIGH', 'registrar': 'NameSilo, LLC', 'country': 'US', 'status': 'success'}]
  Email analysis   : [{'email': 'marketing@zaviyarhayatgroup.com', 'domain': 'zaviyarhayatgroup.com', 'prefix': 'marketing', 'is_free_provider': False, 'suspicious_prefix': True, 'has_mx_records': True, 'risk_signals': ['suspicious prefix pattern: marketing'], 'email_risk': 'MEDIUM', 'status': 'success'}]
  Company existence: {'domain_checked': 'zaviyarhayatgroup.com', 'company_name': 'ZaviyarhayatGroup and Recruitment Team WAS Group | FastInsu', 'name_domain_match': True, 'existence_

Phase 3 v1 launch      → 43% on ADNOC scam (missed)

You observed           → low-signal scams need new tools

Added company checker  → still 33% (tools not running)

You ran diagnostic     → found empty entities

Fixed extraction       → 57% (threshold crossed)

You ran diagnostic     → found existence_risk wrong

Fixed unreachable rule → 80% (strong confident detection)